# Cross-fitted PainNAS on Google Colab

This notebook runs uncertainty-aware neural architecture search on five deterministic outer subject blocks. Each block search excludes the complete block, evaluates candidate architectures with three independent inner subject folds, and maximizes mean subject accuracy minus its standard error. The winning inner-fold checkpoint then warm-starts an individual LOSO continuation for every subject in that block. Each continuation uses a fresh optimizer and all other 86 subjects; the target subject is evaluated only on its predefined `Test` samples.

Select **Runtime → Change runtime type → GPU** before starting. Search databases, winning checkpoints, and completed LOSO folds are stored on Drive and can be resumed after a disconnect.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys
REPO_URL = 'https://github.com/hhihn/FewShotPainAdaptation.git'
PROJECT_DIR = Path('/content/FewShotPainAdaptation')
BRANCH_NAME = 'main'
if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only
%cd $PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
assert (PROJECT_DIR / 'painnas/cross_fitted_loso.py').is_file()


## 2. Install pinned dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r $PROJECT_DIR/painnas/requirements-colab.txt


## 3. Stage BioVid on the local Colab SSD

In [ ]:
from data_loaders.dataset_staging import stage_predefined_dataset_from_archive
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/PainData')
LOCAL_DATA_DIR = Path('/content/PainData')
BIOVID_ROOT = stage_predefined_dataset_from_archive(
    'biovid_part_a',
    drive_data_dir=DRIVE_DATA_DIR,
    local_data_dir=LOCAL_DATA_DIR,
    local_archive_dir=Path('/content'),
)
DATA_DIR = LOCAL_DATA_DIR
print('BioVid root:', BIOVID_ROOT)


## 4. Verify GPU and configure reproducibility

In [ ]:
import random
import numpy as np
import tensorflow as tf
GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Select a Colab GPU runtime.'
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print('TensorFlow:', tf.__version__, 'GPUs:', GPUS)


## 5. Configure cross-fitted NAS and LOSO

The full run performs five block-level NAS studies rather than 87 independent studies. Each trial is evaluated over three inner subject folds. The final continuation starts from the winning fold checkpoint but always creates a new optimizer. Change `RUN_NAME` whenever any configuration value changes; the manifest rejects incompatible resumes.

In [ ]:
from painnas.config import PainNASConfig
RUN_NAME = 'cross_fitted_run_003'
OUTPUT_DIR = Path('/content/drive/MyDrive/PainNAS') / RUN_NAME
CROSS_FITTED_DIR = OUTPUT_DIR / 'cross_fitted_loso'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESUME = True
LOSO_START_INDEX = None  # One-based and inclusive.
LOSO_STOP_INDEX = None   # Set both values to run a resumable chunk.
MAX_FOLDS = None         # Debug-only cap after the index range.
CONFIG = PainNASConfig(
    seed=SEED,
    batch_size=100,
    n_trials=10,
    search_max_epochs=20,
    cross_fitted_continuation_epochs=100,  # Set an integer to override the search-derived epoch count.
    search_patience=8,
    outer_block_count=5,
    inner_fold_count=3,
    uncertainty_beta=1.0,
    max_parameters=20_000_000,
    bootstrap_samples=10_000,
)
print(CONFIG)
print('Output:', CROSS_FITTED_DIR)


## 6. Load BioVid and audit the deterministic subject plan

In [ ]:
import pandas as pd
from painnas.data import load_biovid_binary, build_cross_fitted_subject_plan
ARRAYS = load_biovid_binary(str(DATA_DIR), CONFIG)
SUBJECT_PLAN = build_cross_fitted_subject_plan(
    ARRAYS.unique_subjects,
    outer_block_count=CONFIG.outer_block_count,
    inner_fold_count=CONFIG.inner_fold_count,
    seed=CONFIG.seed,
)
plan_rows = []
for block_index, (block, inner_folds) in enumerate(
    zip(SUBJECT_PLAN.outer_blocks, SUBJECT_PLAN.inner_folds_by_block), start=1
):
    plan_rows.append({
        'outer_block': block_index,
        'outer_subject_count': len(block),
        'development_subject_count': sum(map(len, inner_folds)),
        'inner_fold_sizes': tuple(map(len, inner_folds)),
        'outer_subjects': block,
    })
display(pd.DataFrame(plan_rows))
assert sorted(s for block in SUBJECT_PLAN.outer_blocks for s in block) == sorted(ARRAYS.unique_subjects.tolist())
print('Samples:', len(ARRAYS.y), 'Shape:', ARRAYS.X.shape)


## 7. Inspect the fixed Table 2 baseline

In [ ]:
from painnas.model import ArchitectureSpec, build_early_fusion_model
BASELINE = ArchitectureSpec.baseline()
BASELINE_MODEL = build_early_fusion_model(
    BASELINE,
    input_shape=(ARRAYS.num_modalities, ARRAYS.sequence_length, 1),
    num_classes=CONFIG.num_classes,
)
print(BASELINE)
print('Parameters:', f'{BASELINE_MODEL.count_params():,}')
del BASELINE_MODEL


## 8. Run or resume cross-fitted block NAS

For each required outer block, this runs or resumes one Optuna study, promotes its winning inner-fold checkpoint, and then performs warm-started individual LOSO continuations. A partial LOSO index range automatically reuses an existing block search when more subjects from that block are requested later.

In [ ]:
from painnas.cross_fitted_loso import run_cross_fitted_loso_nas
SUMMARY = run_cross_fitted_loso_nas(
    ARRAYS,
    CONFIG,
    CROSS_FITTED_DIR,
    resume=RESUME,
    start_index=LOSO_START_INDEX,
    stop_index=LOSO_STOP_INDEX,
    max_folds=MAX_FOLDS,
    verbose=1,
)
display(pd.DataFrame([SUMMARY['metrics']['accuracy'], SUMMARY['metrics']['macro_f1']], index=['accuracy', 'macro_f1']))


## 9. Inspect block searches and uncertainty-aware winners

In [ ]:
import json
import matplotlib.pyplot as plt
winner_rows = []
trial_frames = []
for best_path in sorted(CROSS_FITTED_DIR.glob('blocks/block_*/search/best_architecture.json')):
    payload = json.loads(best_path.read_text())
    winner_rows.append({
        'outer_block': payload['outer_block_index'],
        'trial': payload['best_trial_number'],
        'mean_subject_accuracy': payload['best_subject_accuracy_mean'],
        'standard_error': payload['best_subject_accuracy_standard_error'],
        'objective': payload['best_uncertainty_objective'],
        'median_continuation_epoch': payload['median_best_epoch'],
        'warm_start_inner_fold': payload['warm_start_checkpoint_metadata']['inner_fold_index'],
        'parameters': payload['parameter_count'],
    })
    trials_path = best_path.parent / 'trials.csv'
    frame = pd.read_csv(trials_path)
    frame.insert(0, 'outer_block', payload['outer_block_index'])
    trial_frames.append(frame)
winners = pd.DataFrame(winner_rows).sort_values('outer_block')
display(winners)
if not winners.empty:
    ax = winners.plot(x='outer_block', y=['mean_subject_accuracy', 'objective'], marker='o', ylim=(0, 1), title='Uncertainty-aware architecture selection')
    ax.set_ylabel('inner subject accuracy')
    plt.show()
trials = pd.concat(trial_frames, ignore_index=True) if trial_frames else pd.DataFrame()
display(trials.sort_values(['outer_block', 'value'], ascending=[True, False]).head(30))


## 10. Audit isolation, warm starts, and final training subjects

In [ ]:
audit_rows = []
known_subjects = set(map(int, ARRAYS.unique_subjects))
for result_path in sorted(CROSS_FITTED_DIR.glob('folds/fold_*/result.json')):
    payload = json.loads(result_path.read_text())
    target = int(payload['target_subject'])
    outer = set(payload['outer_block_subjects'])
    development = set(payload['nas_development_subjects'])
    sources = set(payload['source_subjects'])
    inner_subjects = [s for fold in payload['inner_folds'] for s in fold]
    checks = {
        'target_in_outer_block': target in outer,
        'outer_excluded_from_nas': not (outer & development),
        'inner_folds_cover_development': set(inner_subjects) == development and len(inner_subjects) == len(set(inner_subjects)),
        'final_training_excludes_only_target': sources == known_subjects - {target},
        'fresh_optimizer': payload['optimizer_initial_iterations'] == 0,
        'warm_checkpoint_exists': Path(payload['warm_start_checkpoint']).is_file(),
        'continuation_complete': payload['continuation_epochs_ran'] == payload['continuation_epochs'],
    }
    assert all(checks.values()), (result_path, checks)
    audit_rows.append({'fold': payload['fold_index'], 'target': target, 'outer_block': payload['outer_block_index'], **checks})
audit = pd.DataFrame(audit_rows)
display(audit)
print('Audited folds:', len(audit))


## 11. Summarize accumulated LOSO classification results

In [ ]:
fold_metrics_path = CROSS_FITTED_DIR / 'fold_metrics.csv'
if fold_metrics_path.exists():
    fold_metrics = pd.read_csv(fold_metrics_path).sort_values('fold_index')
    display(fold_metrics)
    display(fold_metrics[['accuracy', 'macro_f1', 'auroc', 'cross_entropy']].agg(['mean', 'std']))
    ax = fold_metrics.plot(x='fold_index', y=['accuracy', 'macro_f1'], marker='o', ylim=(0, 1), title='Cross-fitted warm-start LOSO performance')
    ax.set_xlabel('LOSO fold')
    ax.set_ylabel('score')
    plt.show()
architecture_path = CROSS_FITTED_DIR / 'architecture_frequencies.csv'
if architecture_path.exists():
    display(pd.read_csv(architecture_path))


## 12. Optional runtime cleanup

In [ ]:
import gc
tf.keras.backend.clear_session()
gc.collect()
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print('Cleanup complete')
